# CBRAIN OASIS Error Analysis (Step 3D)

This notebook performs descriptive error analysis for the frozen Step 3A baseline model on validation subjects.

Scope is descriptive only: no causal claims.

In [ ]:
# 1) Load artifacts for error analysis
from __future__ import annotations

import csv
import json
from collections import defaultdict
from datetime import date
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd]
if cwd.name == 'notebooks':
    candidate_roots.append(cwd.parent)
else:
    candidate_roots.append(cwd / 'projects' / 'cbrain_oasis_cognition')
    candidate_roots.append(cwd.parent)

repo_root = None
for candidate in candidate_roots:
    if (candidate / 'results' / 'modeling' / 'baseline_validation_predictions.csv').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate baseline_validation_predictions.csv')

model_dir = repo_root / 'results' / 'modeling'
cohort_dir = repo_root / 'results' / 'cohort'
docs_path = repo_root / 'docs' / 'error_analysis_step3d.md'

val_pred_path = model_dir / 'baseline_validation_predictions.csv'
baseline_metrics_path = model_dir / 'baseline_metrics.json'
baseline_cohort_path = cohort_dir / 'baseline_cohort_full.csv'
split_subjects_path = cohort_dir / 'split_subjects.json'

def _read_csv(path: Path) -> tuple[list[str], list[dict]]:
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        return list(reader.fieldnames or []), [dict(r) for r in reader]

val_cols, val_rows = _read_csv(val_pred_path)
cohort_cols, cohort_rows = _read_csv(baseline_cohort_path)
baseline_metrics = json.loads(baseline_metrics_path.read_text(encoding='utf-8'))
split_payload = json.loads(split_subjects_path.read_text(encoding='utf-8'))

print(f'Repo root: {repo_root}')
print(f'Validation prediction rows: {len(val_rows)}')

In [ ]:
# 2) Contract checks + join static cohort attributes
required_pred_cols = ['Subject.ID', 'y_true', 'y_probability', 'y_pred_threshold_0_5']
if val_cols != required_pred_cols:
    raise AssertionError(f'Prediction columns mismatch: {val_cols}')

required_cohort_cols = {'Subject.ID', 'Age', 'Gender', 'SES', 'cognitive_impairment'}
missing = required_cohort_cols - set(cohort_cols)
if missing:
    raise AssertionError(f'Missing required columns in baseline cohort: {sorted(missing)}')

if split_payload['overlap_count'] != 0:
    raise AssertionError('Split overlap must remain zero.')

cohort_by_sid = {}
for r in cohort_rows:
    sid = r['Subject.ID']
    if sid in cohort_by_sid:
        raise AssertionError(f'Duplicate Subject.ID in baseline cohort: {sid}')
    cohort_by_sid[sid] = r

val_subjects_expected = sorted(split_payload['validation_subject_ids'])
if len(val_rows) != len(val_subjects_expected):
    raise AssertionError('Validation row count mismatch vs split payload.')

missing_subjects = [sid for sid in val_subjects_expected if sid not in cohort_by_sid]
if missing_subjects:
    raise AssertionError(f'Validation subjects missing in baseline cohort: {missing_subjects}')

def _age_bin(age: int) -> str:
    if age < 70:
        return '<70'
    if age < 80:
        return '70-79'
    return '80+'

def _prob_band(p: float) -> str:
    if p < 0.2:
        return '[0.0,0.2)'
    if p < 0.4:
        return '[0.2,0.4)'
    if p < 0.6:
        return '[0.4,0.6)'
    if p < 0.8:
        return '[0.6,0.8)'
    return '[0.8,1.0]'

error_rows = []
for r in val_rows:
    sid = r['Subject.ID']
    cohort = cohort_by_sid[sid]

    y_true = int(r['y_true'])
    y_pred = int(r['y_pred_threshold_0_5'])
    y_prob = float(r['y_probability'])

    coh_label = int(cohort['cognitive_impairment'])
    if coh_label != y_true:
        raise AssertionError(f'Label mismatch for {sid}: prediction file {y_true} vs cohort {coh_label}')

    age = int(float(cohort['Age']))
    gender = str(cohort['Gender'])
    ses_raw = str(cohort.get('SES', '')).strip()
    ses_missing = 1 if ses_raw == "" else 0

    if y_true == 1 and y_pred == 1:
        outcome = 'TP'
    elif y_true == 0 and y_pred == 0:
        outcome = 'TN'
    elif y_true == 0 and y_pred == 1:
        outcome = 'FP'
    else:
        outcome = 'FN'

    error_rows.append({
        'Subject.ID': sid,
        'y_true': y_true,
        'y_pred_threshold_0_5': y_pred,
        'y_probability': y_prob,
        'outcome_type': outcome,
        'is_error': 1 if outcome in ('FP','FN') else 0,
        'Age': age,
        'age_bin': _age_bin(age),
        'Gender': gender,
        'SES': ses_raw if ses_raw != '' else None,
        'SES_missing': ses_missing,
        'probability_band': _prob_band(y_prob),
        'distance_from_0_5': abs(y_prob - 0.5),
    })

print('Joined error-analysis rows:', len(error_rows))

In [ ]:
# 3) Aggregate descriptive summaries
def _summary(rows: list[dict]) -> dict:
    n = len(rows)
    tp = sum(1 for r in rows if r['outcome_type'] == 'TP')
    tn = sum(1 for r in rows if r['outcome_type'] == 'TN')
    fp = sum(1 for r in rows if r['outcome_type'] == 'FP')
    fn = sum(1 for r in rows if r['outcome_type'] == 'FN')
    pos = tp + fn
    pred_pos = tp + fp
    return {
        'n': n,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'error_count': fp + fn,
        'error_rate': (fp + fn) / n if n else 0.0,
        'positive_rate': pos / n if n else 0.0,
        'predicted_positive_rate': pred_pos / n if n else 0.0,
    }

def _group(rows: list[dict], key: str, order: list[str] | None = None) -> list[dict]:
    buckets = defaultdict(list)
    for r in rows:
        buckets[str(r[key])].append(r)

    keys = order if order is not None else sorted(buckets.keys())
    out = []
    for k in keys:
        if k not in buckets:
            continue
        s = _summary(buckets[k])
        s['group'] = k
        out.append(s)
    return out

overall = _summary(error_rows)
error_type_counts = {t: sum(1 for r in error_rows if r['outcome_type'] == t) for t in ['TP','TN','FP','FN']}

gender_summary = _group(error_rows, 'Gender', order=['F', 'M'])
ses_missing_summary = _group(error_rows, 'SES_missing', order=['0', '1'])
age_bin_summary = _group(error_rows, 'age_bin', order=['<70', '70-79', '80+'])
prob_band_summary = _group(error_rows, 'probability_band', order=['[0.0,0.2)', '[0.2,0.4)', '[0.4,0.6)', '[0.6,0.8)', '[0.8,1.0]'])

baseline_cm = baseline_metrics['validation_metrics']['confusion_matrix']
if (overall['tp'], overall['tn'], overall['fp'], overall['fn']) != (baseline_cm['tp'], baseline_cm['tn'], baseline_cm['fp'], baseline_cm['fn']):
    raise AssertionError('Error-analysis confusion counts do not match baseline metrics.')

print('Overall error rate:', round(overall['error_rate'], 4))
print('FP/FN counts:', overall['fp'], overall['fn'])

In [ ]:
# 4) Materialize Step 3D outputs and report
error_csv_path = model_dir / 'step3d_error_analysis.csv'
summary_json_path = model_dir / 'step3d_error_group_summary.json'

error_fields = ['Subject.ID','y_true','y_pred_threshold_0_5','y_probability','outcome_type','is_error','Age','age_bin','Gender','SES','SES_missing','probability_band','distance_from_0_5']
with error_csv_path.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=error_fields)
    w.writeheader()
    for r in error_rows:
        w.writerow({k: r.get(k) for k in error_fields})

summary_payload = {
    'date': date.today().isoformat(),
    'model_name': baseline_metrics['model_name'],
    'split': 'validation',
    'threshold': 0.5,
    'overall': overall,
    'error_type_counts': error_type_counts,
    'group_summaries': {
        'gender': gender_summary,
        'ses_missing': ses_missing_summary,
        'age_bin': age_bin_summary,
        'probability_band': prob_band_summary,
    },
    'note': 'Descriptive subgroup patterns only; no causal interpretation.',
}

summary_json_path.write_text(json.dumps(summary_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

def _top_errors(rows: list[dict], kind: str, n: int = 5) -> list[dict]:
    filt = [r for r in rows if r['outcome_type'] == kind]
    filt.sort(key=lambda r: r['distance_from_0_5'])
    return [{'Subject.ID': r['Subject.ID'], 'y_probability': r['y_probability'], 'Age': r['Age'], 'Gender': r['Gender'], 'SES_missing': r['SES_missing']} for r in filt[:n]]

top_fp = _top_errors(error_rows, 'FP', n=5)
top_fn = _top_errors(error_rows, 'FN', n=5)

run_date = date.today().isoformat()
docs_text = f"""# Error Analysis (Step 3D)

Date: {run_date}  
Notebook: `notebooks/07_error_analysis_step3d.ipynb`

## Scope

Descriptive error analysis on validation predictions from Step 3A baseline model (threshold=0.5).

## Overall Validation Error Profile

- N={overall["n"]}
- TP={overall["tp"]}, TN={overall["tn"]}, FP={overall["fp"]}, FN={overall["fn"]}
- Error count={overall["error_count"]}, Error rate={overall["error_rate"]:.4f}

## Group Patterns

Gender error rates:
- {', '.join([f"{g['group']}: {g['error_rate']:.3f} (n={g['n']})" for g in gender_summary])}

SES missingness error rates:
- {', '.join([f"SES_missing={g['group']}: {g['error_rate']:.3f} (n={g['n']})" for g in ses_missing_summary])}

Age-bin error rates:
- {', '.join([f"{g['group']}: {g['error_rate']:.3f} (n={g['n']})" for g in age_bin_summary])}

Probability-band error rates:
- {', '.join([f"{g['group']}: {g['error_rate']:.3f} (n={g['n']})" for g in prob_band_summary])}

## Boundary-Adjacent Errors (Closest to 0.5)

Top FPs near decision boundary:
{json.dumps(top_fp, indent=2)}

Top FNs near decision boundary:
{json.dumps(top_fn, indent=2)}

## Interpretation Guardrail

These are descriptive cohort/model diagnostics only.
They indicate where the model struggles, but do not establish causal relationships.

## Step 3D Artifacts

- `results/modeling/step3d_error_analysis.csv`
- `results/modeling/step3d_error_group_summary.json`
"""
docs_path.write_text(docs_text, encoding='utf-8')

print('Wrote:', error_csv_path)
print('Wrote:', summary_json_path)
print('Wrote:', docs_path)